In [ ]:
import json
import shutil
import logging
import warnings
from pathlib import Path

import cv2
from tqdm import tqdm

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)


# ════════════════════════════════════════════════════════════════════════════
# PATHS — đồng bộ 100% với Notebook (Cell 3 + Cell 13)
# ════════════════════════════════════════════════════════════════════════════

# Trỏ thẳng Drive_DIR vào folder processed_data để các hàm bên dưới đọc đúng file Master
DRIVE_DIR = Path("/content/drive/MyDrive/LayoutLM_Project/processed_data")

# Giữ nguyên đường dẫn này để OpenCV tìm đúng ảnh thô sau khi giải nén cục bộ
IMAGE_DIR = Path("/content/LayoutLM_Project/data/data_subset")

# Thư mục lưu ảnh ResNet đã resize trên máy ảo
RESNET_IMG_DIR = Path("/content/resnet_images")

DRIVE_DIR.mkdir(parents=True, exist_ok=True)
RESNET_IMG_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "val", "test"]


# ════════════════════════════════════════════════════════════════════════════
# NHÁNH BERT
# JSONL Master → trích words → join → bert_{split}.jsonl
# ════════════════════════════════════════════════════════════════════════════

def build_bert_split(split: str) -> None:
    """
    Đọc {split}_metadata.jsonl → " ".join(words) → bert_{split}.jsonl

    Schema output:
        {"text": "PARLIAMENT INVOICE Date 2024", "label": "11"}

    Ảnh không có text (words=[]) → text="" vẫn được giữ.
    BERT encode ra [CLS][SEP]+padding — model học được đặc trưng
    "tài liệu không có OCR text", thay vì bỏ đi.
    """
    master = DRIVE_DIR / f"{split}_metadata.jsonl"
    out_f  = DRIVE_DIR / f"bert_{split}.jsonl"

    if not master.exists():
        log.warning("[BERT %s] Không tìm thấy: %s", split, master)
        return

    n_written = 0
    n_empty   = 0

    with open(master, "r", encoding="utf-8") as src, \
         open(out_f,  "w", encoding="utf-8") as dst:

        for line in tqdm(src, desc=f"BERT  {split:5s}", unit="row"):
            line = line.strip()
            if not line:
                continue
            try:
                row   = json.loads(line)
                words = row.get("words", [])
                text  = " ".join(words)   # giữ nguyên reading order từ JSONL Master

                if not text:
                    n_empty += 1

                dst.write(json.dumps(
                    {"text": text, "label": row["label"]},
                    ensure_ascii=False
                ) + "\n")
                n_written += 1

            except (json.JSONDecodeError, KeyError) as e:
                log.warning("Dòng lỗi bỏ qua: %s", e)
                continue

    log.info("[BERT %s] %d rows → %s  (%d rows không có text)",
             split, n_written, out_f.name, n_empty)


# ════════════════════════════════════════════════════════════════════════════
# NHÁNH RESNET50
# JSONL Master → load ảnh thô → resize 224×224 → JPEG → resnet_{split}.jsonl
# ════════════════════════════════════════════════════════════════════════════

def build_resnet_split(split: str) -> None:
    """
    Với mỗi row trong {split}_metadata.jsonl:
        1. Tìm ảnh gốc theo file_name trong IMAGE_DIR
        2. Load thô → BGR→RGB  (KHÔNG CLAHE — đồng bộ với Notebook)
        3. Resize cứng 224×224  (ResNet50 standard input)
        4. Lưu JPEG q95 vào /content/resnet_images/{split}/{class}/xxx.jpg
        5. Ghi resnet_{split}.jsonl

    Tại sao KHÔNG CLAHE?
        Notebook LayoutLMv3 (Cell 6 - train_transforms) đọc ảnh thô bằng
        Image.open().convert("RGB") rồi đưa thẳng vào augmentation/processor.
        Không có bước CLAHE nào.
        → ResNet50 phải nhìn cùng "miền thị giác thô" để fair comparison.

    Tại sao JPEG q95 thay vì PNG?
        40k ảnh × PNG ≈ vài GB → vượt Drive quota.
        JPEG q95 gần lossless nhưng nhẹ hơn ~3×.
    """
    master = DRIVE_DIR / f"{split}_metadata.jsonl"
    out_f  = DRIVE_DIR / f"resnet_{split}.jsonl"

    if not master.exists():
        log.warning("[ResNet %s] Không tìm thấy: %s", split, master)
        return

    # Đọc vào memory để tqdm có total chính xác
    rows = []
    with open(master, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    continue

    n_ok   = 0
    n_fail = 0

    with open(out_f, "w", encoding="utf-8") as dst:
        for row in tqdm(rows, desc=f"ResNet {split:5s}", unit="img"):
            try:
                src_path = IMAGE_DIR / row["file_name"]

                # Load ảnh thô — KHÔNG xử lý gì thêm (đồng bộ Notebook)
                img = cv2.imread(str(src_path))
                if img is None:
                    n_fail += 1
                    continue

                # BGR → RGB
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                # Resize cứng 224×224 — ResNet50 standard
                img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_AREA)

                # Xây dựng đường dẫn output: /content/resnet_images/{split}/{class}/xxx.jpg
                rel_jpg  = Path(row["file_name"]).with_suffix(".jpg")
                dst_path = RESNET_IMG_DIR / rel_jpg
                dst_path.parent.mkdir(parents=True, exist_ok=True)

                # Lưu JPEG q95 — RGB→BGR vì cv2.imwrite cần BGR
                cv2.imwrite(
                    str(dst_path),
                    cv2.cvtColor(img, cv2.COLOR_RGB2BGR),
                    [cv2.IMWRITE_JPEG_QUALITY, 95],
                )

                dst.write(json.dumps(
                    {"resnet_image_path": str(rel_jpg),
                     "label"            : row["label"]},
                    ensure_ascii=False
                ) + "\n")
                n_ok += 1

            except Exception as e:
                log.warning("Lỗi %s: %s", row.get("file_name", "?"), e)
                n_fail += 1
                continue

    log.info("[ResNet %s] OK: %d | Fail: %d → %s",
             split, n_ok, n_fail, out_f.name)


# ════════════════════════════════════════════════════════════════════════════
# ĐÓNG GÓI ZIP — lên Drive để dùng lâu dài
# ════════════════════════════════════════════════════════════════════════════

def zip_resnet_images() -> None:
    """
    Nén /content/resnet_images/ → resnet_images_clean.zip → copy lên Drive.
    """
    zip_local = Path("/content/resnet_images_clean")
    zip_drive = DRIVE_DIR / "resnet_images_clean.zip"

    log.info("Đang nén resnet_images/ → zip...")

    shutil.make_archive(
        base_name = str(zip_local),
        format    = "zip",
        root_dir  = "/content",
        base_dir  = "resnet_images",
    )

    shutil.copy2("/content/resnet_images_clean.zip", str(zip_drive))
    size_mb = zip_drive.stat().st_size / 1024 / 1024
    log.info("ZIP hoàn thành: %s (%.1f MB)", zip_drive, size_mb)


# ════════════════════════════════════════════════════════════════════════════
# CHẠY
# ════════════════════════════════════════════════════════════════════════════

log.info("=" * 55)
log.info("NHÁNH BERT")
log.info("=" * 55)
for split in SPLITS:
    build_bert_split(split)

log.info("=" * 55)
log.info("NHÁNH RESNET50")
log.info("=" * 55)
for split in SPLITS:
    build_resnet_split(split)

log.info("=" * 55)
log.info("ĐÓNG GÓI ZIP")
log.info("=" * 55)
zip_resnet_images()

log.info("HOÀN THÀNH. Output tại: %s", DRIVE_DIR)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Tạo thư mục đích trước để tránh lỗi cấu trúc
!mkdir -p /content/LayoutLM_Project/data

# Tiến hành giải nén ngầm (-q là chạy im lặng, không in danh sách file làm đơ trình duyệt)
!unzip -q "/content/drive/MyDrive/LayoutLM_Project/data_subset.zip" -d /content/LayoutLM_Project/data

In [ ]:
!pip install -q jsonlines

In [ ]:

# Đọc từ:
#   /content/drive/MyDrive/Project_LayoutLM/train_metadata.jsonl   ← JSONL Master
#   /content/drive/MyDrive/Project_LayoutLM/val_metadata.jsonl
#   /content/drive/MyDrive/Project_LayoutLM/test_metadata.jsonl
#   /content/LayoutLM_Project/data/data_subset/{split}/{class}/*.tif  ← ảnh gốc
#     (sau khi giải nén data_subset.zip — giống hệt Cell 2 trong Notebook)
#
# Ghi ra:
#   /content/drive/MyDrive/Project_LayoutLM/
#       bert_train.jsonl / bert_val.jsonl / bert_test.jsonl
#       resnet_train.jsonl / resnet_val.jsonl / resnet_test.jsonl
#   /content/resnet_images/{split}/{class}/xxx.jpg
#   /content/drive/MyDrive/Project_LayoutLM/resnet_images_clean.zip
#
# CHANGELOG so với bản cũ:
#   [1] Paths đồng bộ với Notebook: Project_LayoutLM thay vì LayoutLM_Project
#   [2] IMAGE_DIR = /content/LayoutLM_Project/data/data_subset (Cell 2+3 Notebook)
#   [3] XÓA HOÀN TOÀN apply_clahe() — Notebook đọc ảnh thô, không CLAHE
#   [4] ResNet branch: load thô → resize 224×224 → JPEG q95 (không tiền xử lý thêm)

import json
import shutil
import logging
import warnings
from pathlib import Path

import cv2
from tqdm import tqdm

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
log = logging.getLogger(__name__)


# ════════════════════════════════════════════════════════════════════════════
# PATHS — đồng bộ 100% với Notebook (Cell 3 + Cell 13)
# ════════════════════════════════════════════════════════════════════════════

# Trỏ thẳng Drive_DIR vào folder processed_data để các hàm bên dưới đọc đúng file Master
DRIVE_DIR = Path("/content/drive/MyDrive/LayoutLM_Project/processed_data")

# Giữ nguyên đường dẫn này để OpenCV tìm đúng ảnh thô sau khi giải nén cục bộ
IMAGE_DIR = Path("/content/LayoutLM_Project/data/data_subset")

# Thư mục lưu ảnh ResNet đã resize trên máy ảo
RESNET_IMG_DIR = Path("/content/resnet_images")

DRIVE_DIR.mkdir(parents=True, exist_ok=True)
RESNET_IMG_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "val", "test"]


# ════════════════════════════════════════════════════════════════════════════
# NHÁNH BERT
# JSONL Master → trích words → join → bert_{split}.jsonl
# ════════════════════════════════════════════════════════════════════════════

def build_bert_split(split: str) -> None:
    """
    Đọc {split}_metadata.jsonl → " ".join(words) → bert_{split}.jsonl

    Schema output:
        {"text": "PARLIAMENT INVOICE Date 2024", "label": "11"}

    Ảnh không có text (words=[]) → text="" vẫn được giữ.
    BERT encode ra [CLS][SEP]+padding — model học được đặc trưng
    "tài liệu không có OCR text", thay vì bỏ đi.
    """
    master = DRIVE_DIR / f"{split}_metadata.jsonl"
    out_f  = DRIVE_DIR / f"bert_{split}.jsonl"

    if not master.exists():
        log.warning("[BERT %s] Không tìm thấy: %s", split, master)
        return

    n_written = 0
    n_empty   = 0

    with open(master, "r", encoding="utf-8") as src, \
         open(out_f,  "w", encoding="utf-8") as dst:

        for line in tqdm(src, desc=f"BERT  {split:5s}", unit="row"):
            line = line.strip()
            if not line:
                continue
            try:
                row   = json.loads(line)
                words = row.get("words", [])
                text  = " ".join(words)   # giữ nguyên reading order từ JSONL Master

                if not text:
                    n_empty += 1

                dst.write(json.dumps(
                    {"text": text, "label": row["label"]},
                    ensure_ascii=False
                ) + "\n")
                n_written += 1

            except (json.JSONDecodeError, KeyError) as e:
                log.warning("Dòng lỗi bỏ qua: %s", e)
                continue

    log.info("[BERT %s] %d rows → %s  (%d rows không có text)",
             split, n_written, out_f.name, n_empty)


# ════════════════════════════════════════════════════════════════════════════
# NHÁNH RESNET50
# JSONL Master → load ảnh thô → resize 224×224 → JPEG → resnet_{split}.jsonl
# ════════════════════════════════════════════════════════════════════════════

def build_resnet_split(split: str) -> None:
    """
    Với mỗi row trong {split}_metadata.jsonl:
        1. Tìm ảnh gốc theo file_name trong IMAGE_DIR
        2. Load thô → BGR→RGB  (KHÔNG CLAHE — đồng bộ với Notebook)
        3. Resize cứng 224×224  (ResNet50 standard input)
        4. Lưu JPEG q95 vào /content/resnet_images/{split}/{class}/xxx.jpg
        5. Ghi resnet_{split}.jsonl

    Tại sao KHÔNG CLAHE?
        Notebook LayoutLMv3 (Cell 6 - train_transforms) đọc ảnh thô bằng
        Image.open().convert("RGB") rồi đưa thẳng vào augmentation/processor.
        Không có bước CLAHE nào.
        → ResNet50 phải nhìn cùng "miền thị giác thô" để fair comparison.

    Tại sao JPEG q95 thay vì PNG?
        40k ảnh × PNG ≈ vài GB → vượt Drive quota.
        JPEG q95 gần lossless nhưng nhẹ hơn ~3×.
    """
    master = DRIVE_DIR / f"{split}_metadata.jsonl"
    out_f  = DRIVE_DIR / f"resnet_{split}.jsonl"

    if not master.exists():
        log.warning("[ResNet %s] Không tìm thấy: %s", split, master)
        return

    # Đọc vào memory để tqdm có total chính xác
    rows = []
    with open(master, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    continue

    n_ok   = 0
    n_fail = 0

    with open(out_f, "w", encoding="utf-8") as dst:
        for row in tqdm(rows, desc=f"ResNet {split:5s}", unit="img"):
            try:
                src_path = IMAGE_DIR / row["file_name"]

                # Load ảnh thô — KHÔNG xử lý gì thêm (đồng bộ Notebook)
                img = cv2.imread(str(src_path))
                if img is None:
                    n_fail += 1
                    continue

                # BGR → RGB
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                # Resize cứng 224×224 — ResNet50 standard
                img = cv2.resize(img, (224, 224), interpolation=cv2.INTER_AREA)

                # Xây dựng đường dẫn output: /content/resnet_images/{split}/{class}/xxx.jpg
                rel_jpg  = Path(row["file_name"]).with_suffix(".jpg")
                dst_path = RESNET_IMG_DIR / rel_jpg
                dst_path.parent.mkdir(parents=True, exist_ok=True)

                # Lưu JPEG q95 — RGB→BGR vì cv2.imwrite cần BGR
                cv2.imwrite(
                    str(dst_path),
                    cv2.cvtColor(img, cv2.COLOR_RGB2BGR),
                    [cv2.IMWRITE_JPEG_QUALITY, 95],
                )

                dst.write(json.dumps(
                    {"resnet_image_path": str(rel_jpg),
                     "label"            : row["label"]},
                    ensure_ascii=False
                ) + "\n")
                n_ok += 1

            except Exception as e:
                log.warning("Lỗi %s: %s", row.get("file_name", "?"), e)
                n_fail += 1
                continue

    log.info("[ResNet %s] OK: %d | Fail: %d → %s",
             split, n_ok, n_fail, out_f.name)


# ════════════════════════════════════════════════════════════════════════════
# ĐÓNG GÓI ZIP — lên Drive để dùng lâu dài
# ════════════════════════════════════════════════════════════════════════════

def zip_resnet_images() -> None:
    """
    Nén /content/resnet_images/ → resnet_images_clean.zip → copy lên Drive.
    """
    zip_local = Path("/content/resnet_images_clean")
    zip_drive = DRIVE_DIR / "resnet_images_clean.zip"

    log.info("Đang nén resnet_images/ → zip...")

    shutil.make_archive(
        base_name = str(zip_local),
        format    = "zip",
        root_dir  = "/content",
        base_dir  = "resnet_images",
    )

    shutil.copy2("/content/resnet_images_clean.zip", str(zip_drive))
    size_mb = zip_drive.stat().st_size / 1024 / 1024
    log.info("ZIP hoàn thành: %s (%.1f MB)", zip_drive, size_mb)


# ════════════════════════════════════════════════════════════════════════════
# CHẠY
# ════════════════════════════════════════════════════════════════════════════

log.info("=" * 55)
log.info("NHÁNH BERT")
log.info("=" * 55)
for split in SPLITS:
    build_bert_split(split)

log.info("=" * 55)
log.info("NHÁNH RESNET50")
log.info("=" * 55)
for split in SPLITS:
    build_resnet_split(split)

log.info("=" * 55)
log.info("ĐÓNG GÓI ZIP")
log.info("=" * 55)
zip_resnet_images()

log.info("HOÀN THÀNH. Output tại: %s", DRIVE_DIR)

BERT  train: 31929row [00:03, 8996.37row/s]
BERT  val  : 3991row [00:00, 4285.02row/s]
BERT  test : 3993row [00:00, 4274.92row/s]
ResNet test : 100%|██████████| 3993/3993 [00:34<00:00, 115.38img/s]
